**Import libraries**

In [0]:
from pyspark.sql import functions as F

In [0]:
%sql
select * from parquet.`s3://yellow-taxi-raw-parquet-806121397587-ap-south-2-an/raw-data`
limit 2

-- 43,05,006

In [0]:
%sql
create catalog if not exists dev

In [0]:
%sql
create schema if not exists dev.taxi_db

**Ingest into bronze table**

In [0]:
source_s3_path = "s3://yellow-taxi-raw-parquet-806121397587-ap-south-2-an/raw-data/"
checkpoints_bronze = "s3://yellow-taxi-raw-parquet-806121397587-ap-south-2-an/checkpoints/bronze/"


# Extract from S3 & Load to Bronze Delta table

(spark.readStream
 .format("cloudFiles")
 .option('cloudFiles.format', 'parquet')
 .option('cloudFiles.schemaLocation', checkpoints_bronze)
 .option('cloudFiles.schemaEvolutionMode','addNewColumnsWithTypeWidening')
 .load(source_s3_path)
 .withColumn('ingested_at', F.current_timestamp())
 .writeStream
 .option('mergeSchema', 'true')
 .option('checkpointLocation', checkpoints_bronze)
 .trigger(availableNow=True)
 .toTable("dev.taxi_db.bronze_yellow_taxi")

)


# .option("badRecordsPath", "s3://yellow-taxi-raw-parquet-806121397587-ap-south-2-an/bad-records/")

# SILVER LAYER


**EDA**

In [0]:
%sql
select count(*)
from dev.taxi_db.bronze_yellow_taxi

-- 43,05,006


In [0]:
%sql
select distinct(VendorID)
from dev.taxi_db.bronze_yellow_taxi
-- 4 : 1,2,7,6

In [0]:
%sql
select * from
dev.taxi_db.bronze_yellow_taxi
limit 5

In [0]:
bronze_df = spark.table("dev.taxi_db.bronze_yellow_taxi")


In [0]:
bronze_df.count()

In [0]:
bronze_df.select([
    F.count(F.when(F.col(c).isNull(), 1)).alias(c)
    for c in bronze_df.columns
]).display()

1195482 were empty trips. Drop those rows.

DATA CLEANING

In [0]:
taxi_clean = taxi_clean.dropDuplicates()


In [0]:
taxi_clean = bronze_df.drop('_rescued_data', 'store_and_fwd_flag')

taxi_clean = taxi_clean.dropna(subset=["tpep_pickup_datetime", "tpep_dropoff_datetime", "fare_amount","passenger_count"])

# taxi_clean = taxi_clean.filter(F.col('passenger_count').isNotNull())  --3109524



In [0]:
taxi_clean.count()

In [0]:
taxi_clean.select([
    F.count(F.when(F.col(c).isNull(), 1)).alias(c)
    for c in taxi_clean.columns
]).display()

In [0]:
taxi_clean.filter((F.col('trip_distance') <= 0) | (F.col('total_amount') <= 0)).count()


43049 trips distance were zero or negative, 49352 trips had negative or zero fare(total_amount).


In [0]:
taxi_clean = taxi_clean.filter((F.col('trip_distance') > 0) & (F.col('total_amount') > 0))
taxi_clean.count()    # 3020975

**checking string columns**

In [0]:
string_cols = [c for c, t in taxi_clean.dtypes if t == 'string']
display(string_cols)

Time validation( dropoff > pickup)

In [0]:
taxi_clean = taxi_clean.filter(
    F.col("tpep_dropoff_datetime") > F.col("tpep_pickup_datetime")
)


Trips less than 1000km

In [0]:
taxi_clean = taxi_clean.filter(F.col('trip_distance') < 1000)

negative values

In [0]:
num_cols = taxi_clean.drop("tpep_pickup_datetime","tpep_dropoff_datetime", "ingested_at")

for c in num_cols.columns:
  invalid_count = num_cols.filter(F.col(c) < 0 ).count()

  if invalid_count > 0:
    print(f'col: {c}, invalids : {invalid_count}')

print(f'END')

In [0]:
taxi_clean = taxi_clean.filter(
    (F.col("fare_amount") >= 0) &
    (F.col("trip_distance") >= 0) &
    (F.col("total_amount") >= 0)
)

In [0]:
taxi_clean.limit(2).display()

### trip duration in minutes, trip speed for anamoly detection, type casting, name standardization

**adding trip_minutes**

In [0]:
taxi_clean = (taxi_clean
              .withColumn(
                  'trip_minutes', 
                  F.round((F.unix_timestamp(F.col("tpep_dropoff_datetime")) - F.unix_timestamp(F.col("tpep_pickup_datetime")))/60, 2) )
)

taxi_clean.select('tpep_pickup_datetime', "tpep_dropoff_datetime", 'trip_minutes').limit(5).display()

In [0]:
taxi_clean.groupBy(
    F.when(F.col("trip_minutes") > 1440, "Over 24 Hours")
     .when(F.col("trip_minutes") > 360, "6-24 Hours")
     .otherwise("Normal")
).count().show()

In [0]:
taxi_clean.count()

**Data boundary/Sanity check**

In [0]:
# Calculate mph to find "stagnant" meters or valid trips
taxi_clean=(
taxi_clean.withColumn(
    "avg_speed_mph", 
    F.round(F.col("trip_distance") / (F.col("trip_minutes") / 60), 2)
).filter(
    (F.col("avg_speed_mph") > 2) & (F.col("avg_speed_mph") < 100) & (F.col("trip_minutes") > 1) & (F.col("trip_minutes") < 360)
)
)

In [0]:
taxi_clean.count()

In [0]:
taxi_clean.columns

**Name standardization and type casting**

In [0]:

from pyspark.sql.types import *

# 1. Define the transformation map: { "OldName": ("NewName", DataType()) }
silver_schema = {
    "VendorID": ("vendor_id", IntegerType()),
    "tpep_pickup_datetime": ("pickup_datetime", TimestampNTZType()),
    "tpep_dropoff_datetime": ("dropoff_datetime", TimestampNTZType()),
    "passenger_count": ("passenger_count", IntegerType()),
    "trip_distance": ("trip_distance_miles", DoubleType()),
    "RatecodeID": ("rate_code_id", IntegerType()),
    "PULocationID": ("pickup_location_id", IntegerType()),
    "DOLocationID": ("dropoff_location_id", IntegerType()),
    "payment_type": ("payment_type_id", IntegerType()),
    "fare_amount": ("fare_amount", DecimalType(10, 2)),
    "extra": ("extra_surcharge", DecimalType(10, 2)),
    "mta_tax": ("mta_tax", DecimalType(10, 2)),
    "tip_amount": ("tip_amount", DecimalType(10, 2)),
    "tolls_amount": ("tolls_amount", DecimalType(10, 2)),
    "improvement_surcharge": ("improvement_surcharge", DecimalType(10, 2)),
    "total_amount": ("total_amount", DecimalType(10, 2)),
    "congestion_surcharge": ("congestion_surcharge", DecimalType(10, 2)),
    "Airport_fee": ("airport_fee", DecimalType(10, 2)),
    "cbd_congestion_fee": ("cbd_congestion_fee", DecimalType(10, 2)),
    "ingested_at": ("ingested_at", TimestampType()),
    "trip_minutes": ("duration_minutes", DoubleType()),
    "avg_speed_mph": ("avg_speed_mph", DoubleType())
}

# 2. Execute the rename and cast
taxi_silver = taxi_clean.select([
    F.col(old_name).alias(new_info[0]).cast(new_info[1])
    for old_name, new_info in silver_schema.items()
    if old_name in taxi_clean.columns
])

In [0]:
taxi_silver.limit(2).display()

### write to silver table

In [0]:
target = "dev.taxi_db.silver_yellow_taxi"

(taxi_silver.write
 .format('delta')
 .mode('overwrite')
 .saveAsTable(target)
)


# Incremental ingestion to silver table

# checkpoints_silver = "s3://yellow-taxi-raw-parquet-806121397587-ap-south-2-an/checkpoints/silver/"

# (taxi_silver.writeStream
#  .format('delta')
#  .option("checkpointLocation", checkpoints_silver) # For incremental tracking
#  .outputMode("overwrite")
#  .toTable(target)
# )

In [0]:
# vendor 6 and 7 records are all gone in silver transformation. checking error:


# (bronze_df.select("*")
# .filter(((F.col("VendorID") == 6) | (F.col("VendorID") == 7)) & (F.col("passenger_count").isNotNull() ))
# .display()

# ) 

# UNDER 1 MINUTES TRIPS
under1min = bronze_df.select("*").filter(
    ((F.col("VendorID") == 6) | (F.col("VendorID") == 7)) & 
    (((F.unix_timestamp(F.col("tpep_dropoff_datetime")) - 
       F.unix_timestamp(F.col("tpep_pickup_datetime"))) / 60) < 1)
).count()

null_psngr = bronze_df.filter(F.col("VendorID").isin(6,7) & F.col('passenger_count').isNull()).count()

print(f'{under1min} + {null_psngr} = {under1min + null_psngr}')



> 57423 rows from vendor 6 and 7 are under 1 minutes trips, 4529 rows were null passenger count

In [0]:
bronze_df.filter(F.col("VendorID").isin(6,7)).count()

In [0]:
# Bronze
bronze_df.groupBy("VendorID").count().show()

# Silver
taxi_clean.groupBy("VendorID").count().show()

In [0]:
%sql
select count(*) from dev.taxi_db.silver_yellow_taxi --2910249